# PxFquery Forward Query Demo (Clean Workbench)

This notebook focuses on **forward queries** only (`perturbation -> function`).
It is designed to be readable, minimal, and reproducible for daily resolver checks.

## 1. Environment Setup

This cell sets paths, loads `workspace/.env`, and ensures `PxFquery` is importable.

In [ ]:
import os
import sys
import time
import json
from pathlib import Path

ROOT = Path(r"d:/Projects/7_rush/3_functional_query/1_claude")
WS = ROOT / "workspace"

if str(WS / "script") not in sys.path:
    sys.path.insert(0, str(WS / "script"))

def load_env(env_path: Path) -> None:
    if not env_path.exists():
        print(f"[WARN] .env not found: {env_path}")
        return
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip().strip('\"').strip("'"))

load_env(WS / ".env")
print("Workspace:", WS)
print("MINIMAX_MODEL:", os.getenv("MINIMAX_MODEL", "(not set)"))

## 2. Build Resolver

Use `hybrid_fast` for quick checks, and `always_llm` for full LLM behavior comparisons.

In [ ]:
from PxFquery import PxFquery
from typing import Optional

def build_pxf(resolver_mode: str = "hybrid_fast", provider: str = "minimax", model: Optional[str] = None):
    assert resolver_mode in {"hybrid_fast", "always_llm"}

    pxf = PxFquery()
    pxf.load_data_dir(str(WS / "output/store/gsea_anndata"))
    pxf.enable_resolver(
        index_dir=str(WS / "output/store/query_index"),
        provider=provider,
        model=model or os.getenv("MINIMAX_MODEL", "MiniMax-M2.7"),
        summary_include_numbers=False,
        use_fast_path=(resolver_mode == "hybrid_fast"),
        verbosity="normal",
    )
    resolver = pxf._resolver
    return pxf, resolver

pxf, resolver = build_pxf(resolver_mode="hybrid_fast", provider="minimax")
print("Resolver ready. mode=hybrid_fast")

## 3. Run One Forward Query

Start with one stable case to confirm the pipeline and inspect resolver metadata.

In [ ]:
query = "In A549, what pathways are affected by EGFR knockdown?"

t0 = time.perf_counter()
res = resolver.resolve_and_query(query, top_n=10, summarize=False)
dt = time.perf_counter() - t0

meta = res.resolver_meta or {}
print("elapsed_sec:", round(dt, 3))
print("found:", res.found)
print("hit_level:", meta.get("hit_level"))
print("pert_type:", meta.get("pert_type"))
print("selected_source:", meta.get("selected_source"))
print("cell_resolution_reason:", meta.get("cell_resolution_reason"))


In [ ]:
# Inspect LLM call details for this query (query-level + total-level).
print(json.dumps((res.resolver_meta or {}).get("llm_call_stats", {}), ensure_ascii=False, indent=2))

## 4. Small Batch Demo with Time Budget

This avoids runaway runs: we cap query count and enforce a max total runtime.

In [ ]:
TEST_QUERIES = [
    "In A549, what pathways are affected by EGFR knockdown?",
    "In non-small cell lung carcinoma, what happens if EGFR is suppressed?",
    "In A549, what pathways are affected by an EGFR inhibitor?",
    "In PC3, estimate pathway response for l-theanine-like perturbation.",
]

MAX_QUERIES = 4
MAX_TOTAL_SECONDS = 120

rows = []
t_batch = time.perf_counter()

for i, q in enumerate(TEST_QUERIES[:MAX_QUERIES], start=1):
    if time.perf_counter() - t_batch > MAX_TOTAL_SECONDS:
        print(f"[STOP] Time budget reached at query #{i}.")
        break

    t0 = time.perf_counter()
    r = resolver.resolve_and_query(q, top_n=10, summarize=False)
    dt = time.perf_counter() - t0
    m = r.resolver_meta or {}

    rows.append({
        "query": q,
        "found": bool(r.found),
        "hit_level": m.get("hit_level"),
        "pert_type": m.get("pert_type"),
        "selected_source": m.get("selected_source"),
        "cell_resolution_reason": m.get("cell_resolution_reason"),
        "elapsed_sec": round(dt, 3),
    })

rows

In [ ]:
# Optional pretty table if pandas is available.
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    print(rows)

## 5. Mode Comparison on a Single Query

Use this to compare `hybrid_fast` vs `always_llm` behavior on the same prompt.

In [ ]:
compare_query = "In non-small cell lung carcinoma, what happens if EGFR is suppressed?"

def run_once(mode: str, q: str):
    _, local_resolver = build_pxf(resolver_mode=mode, provider="minimax")
    t0 = time.perf_counter()
    result = local_resolver.resolve_and_query(q, top_n=10, summarize=False)
    dt = time.perf_counter() - t0
    meta = result.resolver_meta or {}
    return {
        "mode": mode,
        "found": bool(result.found),
        "hit_level": meta.get("hit_level"),
        "selected_source": meta.get("selected_source"),
        "cell_resolution_reason": meta.get("cell_resolution_reason"),
        "elapsed_sec": round(dt, 3),
    }

comparison = [run_once("hybrid_fast", compare_query), run_once("always_llm", compare_query)]
comparison

## 6. What to Inspect When Results Look Wrong

For each suspicious query, inspect these fields from `resolver_meta`:

- `hit_level`
- `pert_type` and `selected_source`
- `cell_resolution_reason`
- `requested_cell` / `used_cell`
- `requested_perturbation` / `used_perturbation`
- `proxy_cells_checked` and `proxy_perts_checked`
- `evidence_bundle`
- `llm_call_stats`

These fields together explain whether the issue is parsing, mapping, retrieval, or evidence policy.